# ACCERT AP1000 to IAT China to CRF

This notebook runs the three-tool workflow:

1. Run the ACCERT AP1000 model and generate account CSV files.
2. Convert the ACCERT account output into a CRF/IAT baseline, then run IAT for China.
3. Use the China-adjusted CSV as the CRF baseline to evaluate FOAK-to-NOAK results.

The AP1000 CRF base case uses a well-executed FOAK construction duration of 76 months.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import tempfile

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "Main.py").exists():
    REPO_ROOT = Path.cwd().parents[1]

SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from crf import accert_output_to_crf_baseline, run_one_scenario, results_to_dataframe, print_scenario_result
from crf.io.excel_inputs import InputStore
from iat import level_account_summary, run_adjustment

OUTPUT_DIR = Path(os.environ.get(
    "ACCERT_WORKFLOW_OUTPUT_DIR",
    REPO_ROOT / "tutorial" / "combined" / "outputs" / "ap1000_china_notebook",
))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ACCERT_INPUT = REPO_ROOT / "tutorial" / "accert" / "AP1000.son"
REACTOR_TYPE = "AP1000"
COUNTRY = "China"
YEAR_DOLLAR = 2024
CONSTRUCTION_DURATION_MONTHS = 76

ACCERT_INPUT, OUTPUT_DIR

## 1. Run ACCERT AP1000

The ACCERT run uses a temporary SQLite database copy so the reference database in `src/accertdb.sqlite` is not modified.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    db_path = Path(tmp) / "accertdb.sqlite"
    shutil.copy2(REPO_ROOT / "src" / "accertdb.sqlite", db_path)
    env = {**os.environ, "ACCERT_SQLITE_DB": str(db_path)}
    subprocess.run(
        [sys.executable, str(REPO_ROOT / "src" / "Main.py"), "-i", str(ACCERT_INPUT)],
        cwd=OUTPUT_DIR,
        env=env,
        check=True,
    )

accert_account_csv = sorted(OUTPUT_DIR.glob("ap1000_upd_acc_*.csv"))[-1]
accert_cost_element_csv = sorted(OUTPUT_DIR.glob("ap1000_upd_ce_*.csv"))[-1]
accert_post_csv = sorted(OUTPUT_DIR.glob("ap1000_post_*.csv"))[-1]

print("ACCERT account CSV:", accert_account_csv)
print("ACCERT cost element CSV:", accert_cost_element_csv)
print("ACCERT post-process CSV:", accert_post_csv)

pd.read_csv(accert_account_csv).head(10)

## 2. Convert ACCERT Output and Run IAT for China

The converter maps ACCERT AP1000 accounts into the CRF/IAT account layout, keeps the ACCERT direct-account cost basis, and distributes the user-assigned total 20s labor hours using AP1000 baseline labor-hour proportions.

In [ ]:
ap1000_baseline, _ = InputStore().get_baseline(REACTOR_TYPE)
total_20s_labor_hours = float(
    ap1000_baseline.loc[
        ap1000_baseline["Account"].astype(str).isin(["21", "22", "23", "24", "26"]),
        "Site Labor Hours",
    ].sum()
)

converted_baseline_csv = OUTPUT_DIR / "ap1000_accert_for_iat_crf.csv"
converted_baseline = accert_output_to_crf_baseline(
    accert_account_csv,
    converted_baseline_csv,
    reactor_type=REACTOR_TYPE,
    total_20s_labor_hours=total_20s_labor_hours,
)

print(f"Assigned total 20s labor hours: {total_20s_labor_hours:,.2f}")
print("Converted CRF/IAT baseline CSV:", converted_baseline_csv)
converted_baseline.head(15)

In [ ]:
iat_china_csv = OUTPUT_DIR / "ap1000_accert_china_iat.csv"
iat_result = run_adjustment(
    {
        "reactor_type": "ACCERT output-LR",
        "country": COUNTRY,
        "year_dollar": YEAR_DOLLAR,
        "input_csv": converted_baseline_csv,
        "output_csv": iat_china_csv,
    }
)

print("China-adjusted IAT CSV:", iat_china_csv)
print(f"Input OCC: {iat_result['input_occ']:,.2f}")
print(f"Adjusted OCC: {iat_result['adjusted_occ']:,.2f}")
print(f"Adjustment ratio of OCC: {iat_result['occ_adjustment_ratio']:.4f}")

level_account_summary(iat_result["adjusted_costs"], max_level=2)

## 3. Run CRF with the China-Adjusted Baseline

CRF reads the IAT CSV through `baseline_csv`. Because the IAT output includes adjusted cost columns, CRF uses those adjusted columns as the new baseline.

In [ ]:
crf_config = {
    "reactor_type": REACTOR_TYPE,
    "baseline_csv": str(iat_china_csv),
    "f_22": 250_000_000,
    "f_2321": 150_000_000,
    "land_cost_per_acre_0": 22_000,
    "construction_duration_0": CONSTRUCTION_DURATION_MONTHS,
    "startup_0": 28,
    "staggering_ratio": 0.75,
}

crf_levers = {
    "num_orders": 10,
    "num_NOAK": 8,
    "itc_percent": 0,
    "n_itc": 0,
    "interest_percent": 6,
    "design_completion_percent": 70,
    "design_maturity": 1,
    "proc_exp": 0.5,
    "N_proc": 3,
    "ce_exp": 0.5,
    "N_cons": 5,
    "ae_exp": 0.5,
    "N_AE": 4,
    "standardization_percent": 80,
    "modularity_code": 0,
    "bop_grade_code": 0,
    "rb_grade_code": 0,
}

crf_result = run_one_scenario(crf_config, crf_levers)
print_scenario_result(crf_result)
results_to_dataframe(crf_result)

In [ ]:
summary = pd.DataFrame(
    [
        {
            "ACCERT account CSV": str(accert_account_csv),
            "IAT China CSV for CRF": str(iat_china_csv),
            "FOAK OCC ($/kWe)": crf_result["OCC_1"],
            "NOAK OCC ($/kWe)": crf_result[f"OCC_{crf_levers['num_NOAK']}"],
            "FOAK TCI ($/kWe)": crf_result["TCI_1"],
            "NOAK TCI ($/kWe)": crf_result[f"TCI_{crf_levers['num_NOAK']}"],
            "FOAK construction duration (months)": crf_result["duration_1"],
            "NOAK construction duration (months)": crf_result[f"duration_{crf_levers['num_NOAK']}"],
        }
    ]
)
summary